# MarsLandformNet V3 — End-to-End DINOv2-LoRA Fine-Tuning

**Goal**: Train a tile-level Mars landform classifier achieving **F1 ≥ 0.8**

**Architecture**:
- DINOv2-base (ViT-B/14, 86M params) with LoRA adapters (r=16, alpha=32)
- MOLA topographic features (25-dim) concatenated with CLS token
- MLP classification head → 4 classes: LDA, LVF, CCF, OTHER

**Training Strategy**:
- LoRA adapters initialized from Mars SSL pretraining
- Last 2 transformer blocks unfrozen for fine-tuning
- Strong augmentation: rotation, flip, color jitter
- Class-weighted CE loss (sqrt-inverse frequency)
- AdamW with cosine LR schedule + warmup

**Data**: 58,807 labeled tiles from 639 HiRISE browse images
- Train: 41,195 | Val: 8,905 | Test: 8,707
- Classes: LDA=15069, LVF=2896, CCF=758, OTHER=40084

## 0. Setup & Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate timm pillow scikit-learn matplotlib seaborn

In [ ]:
import os
import json
import time
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f}GB)')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')
    
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

## 1. Download & Extract Training Data

In [ ]:
DATA_DIR = Path('/content/v3_data')
DATA_DIR.mkdir(exist_ok=True)

TAR_URL = 'https://github.com/jejuchild/MarsLab/releases/download/v3-training-e2e/v3_colab_e2e_data.tar.gz'
TAR_PATH = DATA_DIR / 'v3_colab_e2e_data.tar.gz'

if not (DATA_DIR / 'tile_labels_v3.json').exists():
    print('Downloading training data (~740MB)...')
    !wget -q --show-progress -O {TAR_PATH} {TAR_URL}
    print('Extracting...')
    !tar xzf {TAR_PATH} -C {DATA_DIR}
    !rm {TAR_PATH}
    print('Done!')
else:
    print('Data already extracted.')

# Verify
print(f'\nContents of {DATA_DIR}:')
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size/1e6:.1f}MB')
    elif f.is_dir():
        n = sum(1 for _ in f.rglob('*.jpg'))
        print(f'  {f.name}/: {n} JPEGs')

## 2. Configuration

In [ ]:
# ─── Hyperparameters ───────────────────────────────────────────────────────
CFG = {
    # Model
    'model_name': 'facebook/dinov2-base',
    'hidden_dim': 768,       # DINOv2-base CLS token dim
    'mola_dim': 25,          # MOLA feature dimension
    'num_classes': 4,        # LDA, LVF, CCF, OTHER
    'head_hidden': 256,      # MLP head hidden dim
    'dropout': 0.3,          # Classifier dropout
    
    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'lora_targets': ['query', 'key', 'value'],
    'unfreeze_last_n_blocks': 2,  # Unfreeze last N transformer blocks
    
    # Training
    'batch_size': 64,
    'num_epochs': 60,
    'lr_backbone': 5e-5,     # LR for LoRA + unfrozen blocks
    'lr_head': 1e-3,         # LR for classification head
    'weight_decay': 0.01,
    'warmup_epochs': 3,
    'label_smoothing': 0.05,
    
    # Data
    'tile_size': 224,
    'num_workers': 2,
    'class_names': ['LDA', 'LVF', 'CCF', 'OTHER'],
    'class_to_idx': {'LDA': 0, 'LVF': 1, 'CCF': 2, 'OTHER': 3},
    
    # Augmentation
    'aug_hflip': True,
    'aug_vflip': True,
    'aug_rotation': True,     # 0/90/180/270 degrees
    'aug_color_jitter': 0.2,  # brightness & contrast jitter
    
    # Paths
    'data_dir': '/content/v3_data',
    'save_dir': '/content/drive/MyDrive/marslandform_v3',
}

print('Config:')
for k, v in CFG.items():
    print(f'  {k}: {v}')

## 3. Dataset

In [ ]:
class MarsLandformDataset(Dataset):
    """Tile-level Mars landform dataset with MOLA features."""
    
    def __init__(self, tile_labels, split_indices, mola_features, tile_index,
                 data_dir, class_to_idx, transform=None):
        self.data_dir = Path(data_dir)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.mola_features = mola_features
        self.tile_index = tile_index
        
        # Filter to split indices, skip UNLABELED
        self.samples = []
        for idx in split_indices:
            t = tile_labels[idx]
            if t['label'] == 'UNLABELED':
                continue
            self.samples.append(t)
        
        print(f'  Dataset: {len(self.samples)} samples')
        labels = [s['label'] for s in self.samples]
        for cls_name in sorted(class_to_idx.keys()):
            n = sum(1 for l in labels if l == cls_name)
            print(f'    {cls_name}: {n} ({100*n/len(labels):.1f}%)')
    
    def __len__(self):
        return len(self.samples)
    
    def _load_tile_image(self, sample):
        """Load tile JPEG and return as tensor."""
        img_id = sample['image_id']
        tr, tc = sample['tile_row'], sample['tile_col']
        tile_key = f'{img_id}_{tr}_{tc}'
        
        rel_path = self.tile_index.get(tile_key)
        if rel_path:
            img_path = self.data_dir / rel_path
        else:
            img_path = self.data_dir / 'tiles' / img_id / f'tile_{tr:03d}_{tc:03d}.jpg'
        
        try:
            img = Image.open(img_path).convert('RGB')
            img = np.array(img, dtype=np.float32) / 255.0  # HWC, 0-1
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.float32)
        
        return img
    
    def _get_mola(self, sample):
        """Get MOLA features for this tile."""
        img_id = sample['image_id']
        tile_key = f"{sample['tile_row']}_{sample['tile_col']}"
        
        img_mola = self.mola_features.get(img_id, {})
        if tile_key in img_mola:
            return img_mola[tile_key].astype(np.float32)
        return np.zeros(25, dtype=np.float32)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load image (HWC, 0-1 float)
        img = self._load_tile_image(sample)
        
        # Augmentations (applied on numpy HWC)
        if self.transform:
            img = self.transform(img)
        
        # DINOv2 normalization: ImageNet mean/std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        
        # HWC -> CHW tensor
        img_tensor = torch.from_numpy(img.transpose(2, 0, 1))
        
        # MOLA features
        mola = torch.from_numpy(self._get_mola(sample))
        
        # Label
        label = self.class_to_idx[sample['label']]
        
        return img_tensor, mola, label

In [ ]:
class TrainAugmentation:
    """Training augmentations applied on numpy HWC 0-1 images."""
    
    def __init__(self, cfg):
        self.hflip = cfg['aug_hflip']
        self.vflip = cfg['aug_vflip']
        self.rotation = cfg['aug_rotation']
        self.jitter = cfg['aug_color_jitter']
    
    def __call__(self, img):
        # Random 90-degree rotation
        if self.rotation:
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k=k, axes=(0, 1)).copy()
        
        # Random horizontal flip
        if self.hflip and random.random() > 0.5:
            img = np.fliplr(img).copy()
        
        # Random vertical flip
        if self.vflip and random.random() > 0.5:
            img = np.flipud(img).copy()
        
        # Random brightness & contrast jitter
        if self.jitter > 0:
            # Brightness
            brightness_factor = 1.0 + random.uniform(-self.jitter, self.jitter)
            img = img * brightness_factor
            
            # Contrast
            contrast_factor = 1.0 + random.uniform(-self.jitter, self.jitter)
            mean_val = img.mean()
            img = (img - mean_val) * contrast_factor + mean_val
            
            img = np.clip(img, 0.0, 1.0)
        
        return img

In [ ]:
# Load data
data_dir = Path(CFG['data_dir'])

print('Loading labels...')
with open(data_dir / 'tile_labels_v3.json') as f:
    tile_labels = json.load(f)
print(f'  Total tiles: {len(tile_labels)}')

print('Loading splits...')
with open(data_dir / 'tile_splits_v3.json') as f:
    splits = json.load(f)
print(f'  Train: {len(splits["train"])}, Val: {len(splits["val"])}, Test: {len(splits["test"])}')

print('Loading tile index...')
with open(data_dir / 'tile_index.json') as f:
    tile_index = json.load(f)
print(f'  Tile index: {len(tile_index)} entries')

print('Loading MOLA features...')
mola_features = np.load(data_dir / 'mola_features_by_tile.npy', allow_pickle=True).item()
n_mola = sum(len(v) for v in mola_features.values())
print(f'  MOLA: {n_mola} tiles across {len(mola_features)} images')

# Create datasets
print('\nCreating datasets...')
train_aug = TrainAugmentation(CFG)

print('Train:')
train_ds = MarsLandformDataset(
    tile_labels, splits['train'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=train_aug
)
print('Val:')
val_ds = MarsLandformDataset(
    tile_labels, splits['val'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)
print('Test:')
test_ds = MarsLandformDataset(
    tile_labels, splits['test'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)

# Dataloaders
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                         num_workers=CFG['num_workers'], pin_memory=True)

print(f'\nBatches per epoch - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
# Visualize random samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    idx = random.randint(0, len(train_ds) - 1)
    img_t, mola_t, label = train_ds[idx]
    # Undo normalization for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    ax.set_title(f'{CFG["class_names"][label]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Random Training Samples (with augmentation)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Model Architecture

In [ ]:
from transformers import Dinov2Model
from peft import LoraConfig, get_peft_model


class MarsLandformNetV3(nn.Module):
    """End-to-end DINOv2 + LoRA + MOLA classifier."""
    
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        
        # ─── DINOv2 backbone ───────────────────────────────────────────
        print('Loading DINOv2-base from HuggingFace...')
        self.backbone = Dinov2Model.from_pretrained(cfg['model_name'])
        
        # Apply LoRA
        print('Applying LoRA adapters...')
        lora_config = LoraConfig(
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=cfg['lora_targets'],
            bias='none',
        )
        self.backbone = get_peft_model(self.backbone, lora_config)
        self.backbone.print_trainable_parameters()
        
        # Unfreeze last N transformer blocks
        n_blocks = cfg['unfreeze_last_n_blocks']
        if n_blocks > 0:
            n_layers = len(self.backbone.base_model.model.encoder.layer)
            for i in range(n_layers - n_blocks, n_layers):
                layer = self.backbone.base_model.model.encoder.layer[i]
                for param in layer.parameters():
                    param.requires_grad = True
            print(f'Unfroze last {n_blocks} transformer blocks (layers {n_layers-n_blocks}-{n_layers-1})')
        
        # ─── MOLA feature processor ────────────────────────────────────
        self.mola_bn = nn.BatchNorm1d(cfg['mola_dim'])
        self.mola_proj = nn.Linear(cfg['mola_dim'], 64)
        
        # ─── Classification head ───────────────────────────────────────
        combined_dim = cfg['hidden_dim'] + 64  # 768 + 64 = 832
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, cfg['head_hidden']),
            nn.BatchNorm1d(cfg['head_hidden']),
            nn.GELU(),
            nn.Dropout(cfg['dropout']),
            nn.Linear(cfg['head_hidden'], cfg['head_hidden'] // 2),
            nn.BatchNorm1d(cfg['head_hidden'] // 2),
            nn.GELU(),
            nn.Dropout(cfg['dropout'] * 0.5),
            nn.Linear(cfg['head_hidden'] // 2, cfg['num_classes']),
        )
    
    def forward(self, pixel_values, mola_features):
        # DINOv2 forward -> CLS token
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0]  # (B, 768)
        
        # MOLA branch
        mola = self.mola_bn(mola_features)
        mola = F.gelu(self.mola_proj(mola))  # (B, 64)
        
        # Concatenate and classify
        combined = torch.cat([cls_token, mola], dim=1)  # (B, 832)
        logits = self.classifier(combined)  # (B, 4)
        
        return logits

In [ ]:
def load_ssl_lora_weights(model, ssl_path):
    """Load SSL-pretrained LoRA weights into the PEFT model."""
    ckpt = torch.load(ssl_path, map_location='cpu')
    ssl_state = ckpt['lora_state_dict']
    
    # Map SSL keys -> PEFT keys
    # SSL: backbone.base_model.model.encoder.layer.N.attention.attention.{q,k,v}.lora_{A,B}.default.weight
    # PEFT: base_model.model.encoder.layer.N.attention.attention.{q,k,v}.lora_A.default.weight
    mapped = {}
    for ssl_key, tensor in ssl_state.items():
        # Strip leading 'backbone.' prefix
        peft_key = ssl_key.replace('backbone.', '', 1)
        mapped[peft_key] = tensor
    
    # Load into model backbone
    result = model.backbone.load_state_dict(mapped, strict=False)
    
    loaded = len(mapped) - len(result.unexpected_keys)
    print(f'SSL LoRA weights loaded: {loaded}/{len(mapped)} tensors matched')
    if result.unexpected_keys:
        print(f'  Unexpected: {result.unexpected_keys[:5]}')
    
    return model

In [ ]:
# Build model
model = MarsLandformNetV3(CFG)

# Load SSL LoRA weights
ssl_path = data_dir / 'ssl_lora_weights.pt'
if ssl_path.exists():
    model = load_ssl_lora_weights(model, ssl_path)
else:
    print('No SSL LoRA weights found — training from scratch')

model = model.to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params: {total/1e6:.1f}M')
print(f'Trainable: {trainable/1e6:.1f}M ({100*trainable/total:.1f}%)')

## 5. Training Setup

In [ ]:
# ─── Class weights (sqrt-inverse frequency) ────────────────────────────────
train_labels = [s['label'] for s in train_ds.samples]
label_counts = Counter(train_labels)
total_samples = len(train_labels)

class_weights = []
for cls_name in CFG['class_names']:
    count = label_counts.get(cls_name, 1)
    weight = math.sqrt(total_samples / count)
    class_weights.append(weight)

# Normalize so mean weight = 1.0
mean_w = sum(class_weights) / len(class_weights)
class_weights = [w / mean_w for w in class_weights]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print('Class weights (sqrt-inverse, normalized):')
for name, w, c in zip(CFG['class_names'], class_weights, [label_counts.get(n, 0) for n in CFG['class_names']]):
    print(f'  {name}: {w:.3f} (n={c})')

# ─── Loss ──────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CFG['label_smoothing']
)

# ─── Optimizer (different LR for backbone vs head) ─────────────────────────
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'backbone' in name:
        backbone_params.append(param)
    else:
        head_params.append(param)

print(f'\nBackbone trainable params: {sum(p.numel() for p in backbone_params)/1e6:.2f}M')
print(f'Head trainable params: {sum(p.numel() for p in head_params)/1e6:.2f}M')

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_backbone']},
    {'params': head_params, 'lr': CFG['lr_head']},
], weight_decay=CFG['weight_decay'])

# ─── LR Scheduler: Cosine with warmup ─────────────────────────────────────
total_steps = CFG['num_epochs'] * len(train_loader)
warmup_steps = CFG['warmup_epochs'] * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ─── AMP Scaler ───────────────────────────────────────────────────────────
scaler = GradScaler()

print(f'\nTotal training steps: {total_steps}')
print(f'Warmup steps: {warmup_steps}')

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, (images, mola, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        with autocast():
            logits = model(images, mola)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    
    return avg_loss, f1, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for images, mola, labels in loader:
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast():
            logits = model(images, mola)
            loss = criterion(logits, labels)
        
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    
    return avg_loss, f1, acc, all_preds, all_labels

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

save_dir = Path(CFG['save_dir'])
save_dir.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints will be saved to: {save_dir}')

In [ ]:
# ─── Training loop ────────────────────────────────────────────────────────
best_val_f1 = 0.0
patience = 10
patience_counter = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [], 'lr': []}

print(f'\n{"="*70}')
print(f'Training MarsLandformNet V3 for {CFG["num_epochs"]} epochs')
print(f'{"="*70}\n')

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    
    # Train
    train_loss, train_f1, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, device
    )
    
    # Validate
    val_loss, val_f1, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, device
    )
    
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']
    
    # Log history
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Check for improvement
    improved = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        improved = ' ★ BEST'
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'cfg': CFG,
            'class_weights': class_weights,
        }, save_dir / 'best_model.pt')
    else:
        patience_counter += 1
    
    # Print progress
    print(f'Epoch {epoch:3d}/{CFG["num_epochs"]} | '
          f'Train L={train_loss:.4f} F1={train_f1:.4f} Acc={train_acc:.4f} | '
          f'Val L={val_loss:.4f} F1={val_f1:.4f} Acc={val_acc:.4f} | '
          f'LR={current_lr:.2e} | {elapsed:.0f}s{improved}')
    
    # Per-class F1 every 10 epochs
    if epoch % 10 == 0:
        per_class_f1 = f1_score(val_labels, val_preds, average=None)
        for i, name in enumerate(CFG['class_names']):
            print(f'    {name}: F1={per_class_f1[i]:.4f}')
    
    # Save periodic checkpoint
    if epoch % 20 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'cfg': CFG,
        }, save_dir / f'checkpoint_epoch{epoch}.pt')
        print(f'    Saved checkpoint_epoch{epoch}.pt')
    
    # Early stopping
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)')
        break

print(f'\n{"="*70}')
print(f'Training complete! Best val F1: {best_val_f1:.4f}')
print(f'{"="*70}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(history['train_f1'], label='Train', linewidth=2)
axes[1].plot(history['val_f1'], label='Val', linewidth=2)
axes[1].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].set_title('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# LR
axes[2].plot(history['lr'], linewidth=2, color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved training_curves.png')

## 8. Test Set Evaluation

In [ ]:
# Load best model
best_ckpt = torch.load(save_dir / 'best_model.pt', map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f'Loaded best model from epoch {best_ckpt["epoch"]} (val F1={best_ckpt["val_f1"]:.4f})')

# Evaluate on test set
test_loss, test_f1, test_acc, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

print(f'\n{"="*50}')
print(f'TEST SET RESULTS')
print(f'{"="*50}')
print(f'Loss: {test_loss:.4f}')
print(f'Macro F1: {test_f1:.4f}')
print(f'Accuracy: {test_acc:.4f}')
print(f'{"="*50}\n')

# Classification report
print(classification_report(test_labels, test_preds, 
                            target_names=CFG['class_names'], digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax1)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix (Counts)')

# Percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax2)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig(save_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Export Model for MarsLab Server

In [ ]:
# Save deployment checkpoint (smaller — no optimizer state)
deploy_state = {
    'model_state_dict': model.state_dict(),
    'cfg': CFG,
    'class_names': CFG['class_names'],
    'class_to_idx': CFG['class_to_idx'],
    'test_f1': test_f1,
    'test_acc': test_acc,
    'epoch': best_ckpt['epoch'],
    'class_weights': class_weights,
}
deploy_path = save_dir / 'marslandform_v3_deploy.pt'
torch.save(deploy_state, deploy_path)
print(f'Deployment checkpoint saved: {deploy_path}')
print(f'Size: {deploy_path.stat().st_size / 1e6:.1f}MB')
print(f'\nTest F1: {test_f1:.4f}')
print(f'Test Acc: {test_acc:.4f}')

print(f'\n{"="*60}')
print(f'TRAINING COMPLETE!')
print(f'Best model saved at: {save_dir}/best_model.pt')
print(f'Deploy model saved at: {save_dir}/marslandform_v3_deploy.pt')
print(f'\nTo download: Right-click in file browser → Download')
print(f'Or copy from Google Drive: {CFG["save_dir"]}')
print(f'{"="*60}')

## 10. Error Analysis (Optional)

Examine misclassified tiles to understand failure modes.

In [ ]:
# Find misclassified samples
misclassified = []
for i, (pred, true) in enumerate(zip(test_preds, test_labels)):
    if pred != true:
        misclassified.append((i, pred, true))

print(f'Misclassified: {len(misclassified)}/{len(test_preds)} ({100*len(misclassified)/len(test_preds):.1f}%)')

# Show examples of each confusion type
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
random.shuffle(misclassified)
for i, ax in enumerate(axes.flat):
    if i >= len(misclassified):
        ax.axis('off')
        continue
    idx, pred, true = misclassified[i]
    img_t, mola_t, _ = test_ds[idx]
    
    # Undo normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    
    ax.imshow(img_display)
    ax.set_title(f'True:{CFG["class_names"][true]}\nPred:{CFG["class_names"][pred]}',
                 fontsize=9, color='red')
    ax.axis('off')

plt.suptitle('Misclassified Tiles', fontsize=14)
plt.tight_layout()
plt.savefig(save_dir / 'error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()